# Search strategy comparison

The [hyperparameter search](hyperparameter-search.ipynb) chapter introduced grid,
random, and TPE search. Here we put them head-to-head on the **same objective**
with the **same budget** (number of evaluations), measuring both **quality**
(best error found) and **cost** (wall-clock time) — because a search strategy is
only worth its complexity if it earns back the time it spends. All three run
through [`hyperopt-rs`](https://crates.io/crates/hyperopt-rs) with only the
**sampler** swapped, so the comparison is genuinely apples-to-apples.

In [ ]:
:dep hyperopt-rs = { version = "0.1.1" }
use hyperopt_rs::prelude::*;
use std::time::Instant;

// Synthetic validation error (lower is better), minimum near x = 2.
fn objective(x: f64) -> f64 { (x - 2.0).powi(2) * 0.1 + 0.05 + 0.03 * (x * 4.0).sin() }

let budget = 40usize;

// Best-error-so-far after each trial (an 'any-time' curve) from a finished study.
fn best_so_far(trials: &[Trial]) -> Vec<f64> {
    let mut b = f64::INFINITY;
    trials.iter().map(|t| { if let Some(v) = t.value { b = b.min(v); } b }).collect()
}

// Every strategy runs the SAME define-by-run objective through hyperopt-rs — only
// the sampler differs. Grid needs its points enumerated; random/TPE just the range.
let (grid_curve, grid_time): (Vec<f64>, std::time::Duration) = {
    let t = Instant::now();
    let grid = GridSampler::new().add_float_grid("x", &(0..budget).map(|i| 5.0 * i as f64 / (budget - 1) as f64).collect::<Vec<f64>>());
    let n = grid.grid_size();
    let s = StudyBuilder::new("grid").direction(Direction::Minimize).sampler(grid).build().unwrap();
    s.optimize(|t| { let x = t.suggest_float("x", 0.0, 5.0); Ok(objective(x)) }, n).unwrap();
    (best_so_far(&s.trials().unwrap()), t.elapsed())
};
let (random_curve, random_time): (Vec<f64>, std::time::Duration) = {
    let t = Instant::now();
    let s = StudyBuilder::new("random").direction(Direction::Minimize).sampler(RandomSampler::seeded(0)).build().unwrap();
    s.optimize(|t| { let x = t.suggest_float("x", 0.0, 5.0); Ok(objective(x)) }, budget).unwrap();
    (best_so_far(&s.trials().unwrap()), t.elapsed())
};
let (tpe_curve, tpe_time): (Vec<f64>, std::time::Duration) = {
    let t = Instant::now();
    let s = StudyBuilder::new("tpe").direction(Direction::Minimize).sampler(TpeSampler::seeded(0)).build().unwrap();
    s.optimize(|t| { let x = t.suggest_float("x", 0.0, 5.0); Ok(objective(x)) }, budget).unwrap();
    (best_so_far(&s.trials().unwrap()), t.elapsed())
};

println!("{:<8}  {:>9}  {:>12}", "strategy", "best err", "time");
println!("{:<8}  {:>9.4}  {:>12?}", "grid",   grid_curve[budget - 1],   grid_time);
println!("{:<8}  {:>9.4}  {:>12?}", "random", random_curve[budget - 1], random_time);
println!("{:<8}  {:>9.4}  {:>12?}", "tpe",    tpe_curve[budget - 1],    tpe_time);

## Any-time performance

The chart below plots **best error found so far** against **number of
evaluations** for each strategy (grid = red, random = blue, TPE = green). A curve
that drops faster reaches a good answer with fewer evaluations — the property
that matters when each evaluation is an expensive model fit:

In [ ]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use plotters::prelude::*;

evcxr_figure((520, 360), |root| {
    root.fill(&WHITE)?;
    let max_y = grid_curve[0].max(random_curve[0]).max(tpe_curve[0]) * 1.05;
    let mut chart = ChartBuilder::on(&root)
        .caption("Best error vs evaluations (grid=red, random=blue, tpe=green)", ("sans-serif", 15))
        .margin(10).x_label_area_size(35).y_label_area_size(45)
        .build_cartesian_2d(0f64..budget as f64, 0f64..max_y)?;
    chart.configure_mesh().x_desc("evaluations").y_desc("best error").draw()?;
    chart.draw_series(LineSeries::new((0..budget).map(|i| (i as f64, grid_curve[i])), &RED))?;
    chart.draw_series(LineSeries::new((0..budget).map(|i| (i as f64, random_curve[i])), &BLUE))?;
    chart.draw_series(LineSeries::new((0..budget).map(|i| (i as f64, tpe_curve[i])), &GREEN))?;
    Ok(())
})

## When to use which

Cost and quality together give practical guidance:

- **Grid search** — small, low-dimensional spaces where being exhaustive is
  affordable and reproducibility matters. Cheap per evaluation, but the number of
  points explodes with dimensions.
- **Random search** — a strong, simple default for larger / higher-dimensional
  spaces; near-zero overhead and usually better any-time performance than grid.
- **TPE (Bayesian)** — when each evaluation is *expensive* (e.g. a full
  cross-validated model fit), so it's worth spending extra bookkeeping time to
  need fewer evaluations. Note in the table that TPE's per-evaluation overhead
  makes it the slowest here on a *cheap* objective — that overhead only pays off
  when the objective itself is costly.

```{note}
**Ecosystem maturity.** Rust's HPO tooling has caught up considerably:
[`hyperopt-rs`](https://crates.io/crates/hyperopt-rs) (used here) is an
Optuna-shaped framework — interchangeable samplers (grid / random / TPE /
CMA-ES), pruning, a persistence layer, and both local-parallel and multi-machine
distributed execution — rather than a single-algorithm crate. It's newer and
single-author, so re-verify its state before relying on it. To parallelize the
search itself, `hyperopt-rs` offers `Study::optimize_parallel` (rayon) and a
distributed coordinator/worker mode; see also the
[Multithreading chapter](../04b-multithreading/parallel-ml.ipynb).
```